In [7]:
import math
from pathlib import Path

import numpy as np
import polars as pl
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from fx_forecasting.config import load_experiment_config
from fx_forecasting.data import build_model_frame, build_point_in_time_panel, load_all_datasets
from fx_forecasting.validation import (
    build_validation_folds,
    development_validation_frame,
    fold_frames,
)

ROOT = Path("..")
config = load_experiment_config(ROOT / "configs" / "experiment.yaml")
model_frame = build_model_frame(build_point_in_time_panel(load_all_datasets(config)))
tuning_frame = development_validation_frame(model_frame, config.periods)
folds = build_validation_folds(tuning_frame, config.periods, config.purge_business_days)
PURGE = config.purge_business_days

ALPHA_GRID = (0.01, 0.1, 1.0, 10.0, 100.0, 1000.0)
MINIMUM_MAE_IMPROVEMENT = 0.00001
INNER_SPLITS = 3
OPTIONAL_GROUPS = {
    "rates": (
        "us_rate",
        "us_rate_change_1",
        "jp_rate",
        "jp_rate_change_1",
        "rate_differential",
        "rate_differential_change_1",
    ),
    "equities_and_risk": ("nikkei_return_1", "vix", "vix_change_1"),
    "oil": ("wti_return_1",),
    "macroeconomic_activity": (
        "us_cpi_growth",
        "us_indpro_growth",
        "us_curve_10y2y",
        "us_curve_10y2y_change_1",
    ),
}


In [8]:
def available(frame, features):
    return [
        feature
        for feature in features
        if feature in frame.columns and np.isfinite(frame[feature].to_numpy()).any()
    ]


def inner_splits(height):
    starts = [int((index + 1) * height / (INNER_SPLITS + 1)) for index in range(INNER_SPLITS)]
    stops = [*starts[1:], height]
    return [(start - PURGE, start, stop) for start, stop in zip(starts, stops, strict=True)]


def fit_ridge(train, features, alpha):
    pipeline = make_pipeline(
        SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=alpha)
    )
    return pipeline.fit(train.select(features).to_numpy(), train["target_return"].to_numpy())


def inner_mae(training, features, alpha):
    errors = []
    for train_end, start, stop in inner_splits(training.height):
        train, scored = training[:train_end], training[start:stop]
        usable = available(train, features)
        predicted = fit_ridge(train, usable, alpha).predict(scored.select(usable).to_numpy())
        errors.append(np.mean(np.abs(predicted - scored["target_return"].to_numpy())))
    return float(np.mean(errors))


def historical_mean_mae(training):
    errors = []
    for train_end, start, stop in inner_splits(training.height):
        forecast = training[:train_end]["target_return"].mean()
        errors.append(np.mean(np.abs(training[start:stop]["target_return"].to_numpy() - forecast)))
    return float(np.mean(errors))


def best_alpha(training, features):
    best = (ALPHA_GRID[0], math.inf)
    for alpha in ALPHA_GRID:
        mae = inner_mae(training, features, alpha)
        if mae < best[1] - 1e-12:
            best = (alpha, mae)
    return best


def rank_groups(training, groups):
    everything = available(
        training, dict.fromkeys(feature for group in groups.values() for feature in group)
    )
    alpha, _ = best_alpha(training, everything)
    coefficients = dict(
        zip(everything, fit_ridge(training, everything, alpha)[-1].coef_, strict=True)
    )
    size = {
        name: np.mean([abs(coefficients[feature]) for feature in features])
        for name, features in groups.items()
    }
    return sorted(groups, key=lambda name: (-size[name], name))

## Run the selection on every validation fold

`baseline_mae` is the inner cross-validation MAE of the historical mean, which uses no features. `selected_alpha` is empty when no group is kept. `selected_groups` lists the optional groups the selection kept.

In [9]:
selection_rows = []
group_rows = []
for fold in folds:
    training, _ = fold_frames(tuning_frame, fold)
    groups = {name: available(training, features) for name, features in OPTIONAL_GROUPS.items()}
    groups = {name: features for name, features in groups.items() if features}

    baseline_mae = historical_mean_mae(training)
    selected, kept, alpha, mae = [], [], None, baseline_mae
    for name in rank_groups(training, groups):
        candidate = list(dict.fromkeys([*selected, *groups[name]]))
        candidate_alpha, candidate_mae = best_alpha(training, candidate)
        if candidate_mae < mae - MINIMUM_MAE_IMPROVEMENT:
            selected, kept = candidate, [*kept, name]
            alpha, mae = candidate_alpha, candidate_mae

    for name, features in groups.items():
        group_alpha, group_mae = best_alpha(training, features)
        group_rows.append(
            {
                "refit_id": fold.refit_id,
                "group": name,
                "alpha": group_alpha,
                "mae_improvement": baseline_mae - group_mae,
            }
        )

    selection_rows.append(
        {
            "refit_id": fold.refit_id,
            "training_end": fold.training_end,
            "baseline_mae": baseline_mae,
            "selected_alpha": alpha,
            "selection_mae": mae,
            "selected_groups": ", ".join(kept) or "(none)",
        }
    )

selection = pl.DataFrame(selection_rows)
group_tests = pl.DataFrame(group_rows)
selection

refit_id,training_end,baseline_mae,selected_alpha,selection_mae,selected_groups
str,date,f64,null,f64,str
"""refit-2022-01""",2021-12-29,0.003283,null,0.003283,"""(none)"""
"""refit-2022-02""",2022-01-28,0.003281,null,0.003281,"""(none)"""
"""refit-2022-03""",2022-02-25,0.003256,null,0.003256,"""(none)"""
"""refit-2022-04""",2022-03-30,0.003271,null,0.003271,"""(none)"""
"""refit-2022-05""",2022-04-28,0.003314,null,0.003314,"""(none)"""
…,…,…,…,…,…
"""refit-2023-08""",2023-07-28,0.003653,null,0.003653,"""(none)"""
"""refit-2023-09""",2023-08-30,0.00365,null,0.00365,"""(none)"""
"""refit-2023-10""",2023-09-28,0.003648,null,0.003648,"""(none)"""
